# 03 — Submission and verification
Use locally trained bundles to generate the exact soft-vote submission. First run `python run.py train --data ../data` from the project folder. This notebook performs prediction and checks; it never uploads a submission.

In [ ]:
import os
from pathlib import Path
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[name] = '1'
import numpy as np
import pandas as pd
from copilote_b.config import load_config
from copilote_b.pipeline import initialize, predict, verify
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/copilote_b').is_dir())
default_data = PROJECT.parent / 'data' if (PROJECT.parent / 'data/train.csv').exists() else PROJECT / 'data'
DATA = Path(os.environ.get('COPILOTE_DATA', default_data)).resolve()
OUTPUT = Path(os.environ.get('COPILOTE_OUTPUT', PROJECT / 'outputs/reproduction')).resolve()
config = load_config(PROJECT / 'configs/solution.json')
missing = [name for name in config['members'] if not (OUTPUT / 'models' / f'{name}.joblib').exists()]
if missing:
    raise FileNotFoundError(f'Train these model bundles first: {missing}')

## Generate B
Prediction loads one bundle at a time, checks a common class order, conditions scores on training-only browser support, and applies the frozen weights. It validates the sample's IDs, row count, label set and missing values.

In [ ]:
initialize(DATA, OUTPUT, config)
submission = predict(DATA, config, OUTPUT)
display(submission.head())
print(submission.shape, OUTPUT / 'submission_B.csv')
display(verify(OUTPUT))

## Inspect a soft vote
Row 66 is the sole A/B disagreement in the historical data. Inspecting model outputs does not establish its true label. Values below are uncalibrated softmax scores, not confidence estimates.

In [ ]:
with np.load(OUTPUT / 'probabilities.npz') as saved:
    probabilities, classes = saved['probabilities'], saved['classes']
row = min(65, len(submission) - 1)
top = np.argsort(probabilities[row])[-3:][::-1]
contributions = []
for name, weight in config['weights'].items():
    values = np.load(OUTPUT / f'test_{name}.npy')
    contributions.append({'member': name, 'weight': weight, 'member_winner': classes[values[row].argmax()], **{str(classes[j]): float(weight * values[row, j]) for j in top}})
display(pd.DataFrame(contributions))
display(pd.Series(probabilities[row, top], index=classes[top], name=f'RowId={row + 1} blended score'))

## Optional historical-reference check
These files are used only when the original repository artifacts are available. A standalone copy can skip this cell. Reference verification checks member scores and the final blend at absolute tolerance `1e-12`, along with exact CSV labels and bytes.

In [ ]:
reference_folder = PROJECT.parent / 'artifacts/round3'
if (reference_folder / 'submission_B.csv').exists():
    display(verify(OUTPUT, reference_folder / 'submission_B.csv', reference_folder))
else:
    print('Historical reference not present; internal submission checks already passed.')

## Result
The generated CSV is ready for manual inspection. The original B received a user-reported public score of 0.999; a recreation's integrity is established by the numerical reference check, not by repeating a leaderboard claim. See [the README](../README.md) for CLI reproduction and output descriptions.